# PDF → Curated Text Dataset with NVIDIA NeMo Curator

This notebook turns raw PDF files into a clean, quality-filtered text dataset (JSONL) that can be used for LLM training or fine-tuning.

## Pipeline overview

```
 PDF files ──► Extract text ──► Split into samples ──► Quality filter & clean ──► Curated JSONL
 (raw_pdf)     (PyMuPDF)        (regex on "Sample N:")  (NeMo Curator pipeline)    (curated/)
```

| Step | What happens | Tool |
|------|--------------|------|
| 1. Setup | Install NeMo Curator, mount Google Drive, create folders | `uv`, `nemo-curator` |
| 2. Extract | Read every PDF and write one JSON record per PDF | PyMuPDF |
| 3. Split | Break the extracted text into individual samples | Python `re` |
| 4. Curate | Filter low-quality text and normalise the rest | NeMo Curator `Pipeline` |
| 5. Inspect | Count and preview the final records | Python `json` |

**Before you run it:** put your PDF file(s) in `MyDrive/nemo_curator/data/raw_pdf/` on Google Drive (the folder is created in step 1). A GPU is not required; this notebook installs the CPU text extra (`nemo-curator[text_cpu]`).

## Step 1 – Install NeMo Curator

NeMo Curator is NVIDIA's open-source toolkit for preparing large text datasets. The cell below installs `uv` (a fast Python package installer) and then installs `nemo-curator[text_cpu]` together with its dependencies (Ray, `cosmos-xenna`, `fasttext`, `ftfy`, etc.).

> The install takes a few minutes. The `source ... env` and `source .venv/bin/activate` lines have no lasting effect in Colab (each `!` command runs in its own shell), so the packages end up in the notebook's own Python environment.

In [1]:
!curl -LsSf https://astral.sh/uv/0.12.1/install.sh | sh
!source $HOME/.local/bin/env
!uv venv
!source .venv/bin/activate
!curl -O https://raw.githubusercontent.com/NVIDIA-NeMo/Curator/main/requirements/text_cuda12-overrides.txt
!uv pip install \
  --override text_cuda12-overrides.txt \
  --torch-backend cu129 \
  --extra-index-url https://wheels.vllm.ai/0.22.0/cu129 \
  "nemo-curator[text_cpu]"


downloading uv 0.12.1 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/bin/bash: line 1: /root/.local/bin/env: No such file or directory
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   329  100   329    0     0   1672      0 --:--:-- --:--:-- --:--:--  1678
Using Python 3.13.16 environment at: /usr
Resolved 143 packages in 4.09s
Prepared 42 packages in 1m 27s
Uninstalled 5 packages in 648ms
Installed 42 packages in 293ms
 - absl-py==1.4.0
 + absl-py==2.5.1
 + aiohttp-cors==0.8.1
 + cattrs==26.2.1
 + colorful==0.5.8
 + comment-parser==1.2.5
 + cosmos-xenna==0.2.0
 + courlan==1.4.0
 + dateparser==1.4.3
 + distlib==0.4.4
 + fasttext==0.9.3
 + fastwarc==1.0.9
 + ftfy==6.3.1
 + htmldate==1.10.0
 - huggin

## Step 2 – Mount Google Drive and define folders

Google Drive is used for persistent storage, so your PDFs and results survive after the Colab session ends. All paths are defined once in one place and reused by later cells:

| Variable | Purpose |
|----------|---------|
| `RAW_PDF_DIR` | Input – put your `.pdf` files here |
| `EXTRACTED_JSONL_FILE` | Output of Step 3 – one JSON record per PDF |
| `SPLIT_JSON_FILE` | Output of Step 4 – one JSON record per sample |
| `CURATED_JSONL_FILE` | Output of Step 5 – the final cleaned dataset |

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
from pathlib import Path

# Root directory
BASE_DIR = Path("/content/drive/MyDrive/nemo_curator")

# Directory paths
RAW_PDF_DIR = BASE_DIR / "data" / "raw_pdf"
EXTRACTED_JSONL_DIR = BASE_DIR / "data" / "extracted_jsonl"
CURATED_DIR = BASE_DIR / "data" / "curated"
SPLIT_DIR = BASE_DIR / "data" / "extracted_jsonl" / "split_extracted"

# Fixed file paths
EXTRACTED_JSONL_FILE = EXTRACTED_JSONL_DIR / "pdf_extracted.jsonl"
CURATED_JSONL_FILE = CURATED_DIR / "pdf_cleaned"
SPLIT_JSON_FILE =  SPLIT_DIR / "pdf_extracted_split.jsonl"

# Create directories
for directory in [
    RAW_PDF_DIR,
    EXTRACTED_JSONL_DIR,
    CURATED_DIR,
    SPLIT_DIR,

]:
    directory.mkdir(parents=True, exist_ok=True)
    print("Ready:", directory)

print("\nFixed file paths:")
print("PDF input:       ", RAW_PDF_DIR)
print("Extracted JSONL: ", EXTRACTED_JSONL_FILE)
print("Curated output:  ", CURATED_JSONL_FILE)
print("Split output:  ", SPLIT_JSON_FILE)

Ready: /content/drive/MyDrive/nemo_curator/data/raw_pdf
Ready: /content/drive/MyDrive/nemo_curator/data/extracted_jsonl
Ready: /content/drive/MyDrive/nemo_curator/data/curated
Ready: /content/drive/MyDrive/nemo_curator/data/extracted_jsonl/split_extracted

Fixed file paths:
PDF input:        /content/drive/MyDrive/nemo_curator/data/raw_pdf
Extracted JSONL:  /content/drive/MyDrive/nemo_curator/data/extracted_jsonl/pdf_extracted.jsonl
Curated output:   /content/drive/MyDrive/nemo_curator/data/curated/pdf_cleaned
Split output:   /content/drive/MyDrive/nemo_curator/data/extracted_jsonl/split_extracted/pdf_extracted_split.jsonl


## Step 3 – Extract text from PDFs

PyMuPDF reads each PDF page by page and the pages are joined with blank lines. Each PDF becomes **one** JSON line (JSONL format) with these fields:

- `id` – the PDF filename without extension
- `text` – the full extracted text
- `source_file` – the original filename
- `page_count` – number of pages



In [5]:
%pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 37.3 MB/s eta 0:00:00


In [6]:
# Extract PDF text and save it as JSONL

import json
import fitz  # PyMuPDF
from pathlib import Path

# Input directory containing original PDF files
PDF_DIR = RAW_PDF_DIR

# Fixed output file path
jsonl_path = EXTRACTED_JSONL_FILE

# Find all PDF files
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

# Stop if no PDFs are found
if not pdf_files:
    raise FileNotFoundError(
        f"No PDF files found in {PDF_DIR}. Upload your PDFs first."
    )

# Ensure output directory exists
jsonl_path.parent.mkdir(parents=True, exist_ok=True)

successful = 0
failed = []

# Open in write mode ("w") to overwrite the previous output
with jsonl_path.open("w", encoding="utf-8") as output:
    for pdf_path in pdf_files:
        try:
            with fitz.open(pdf_path) as pdf:
                page_texts = [
                    page.get_text("text") for page in pdf
                ]
                page_count = len(pdf)

            text = "\n\n".join(page_texts).strip()

            # Skip PDFs without extractable text
            if not text:
                print(f"No extractable text: {pdf_path.name}")
                failed.append(pdf_path.name)
                continue

            # One JSON object per PDF
            record = {
                "id": pdf_path.stem,
                "text": text,
                "source_file": pdf_path.name,
                "page_count": page_count,
            }

            output.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            successful += 1
            print(
                f"Extracted: {pdf_path.name} "
                f"({len(text.split())} words)"
            )

        except Exception as e:
            failed.append(pdf_path.name)
            print(f"Failed: {pdf_path.name}: {e}")

print("\nPDF files found:", len(pdf_files))
print("Successfully extracted:", successful)
print("Failed or empty:", len(failed))
print("JSONL saved to:", jsonl_path)

if failed:
    print("Files needing attention:", failed)


Extracted: large_completely_mixed_test.pdf (8652 words)

PDF files found: 1
Successfully extracted: 1
Failed or empty: 0
JSONL saved to: /content/drive/MyDrive/nemo_curator/data/extracted_jsonl/pdf_extracted.jsonl


## Step 4 – Split the document into samples

The test PDF contains many independent text samples, each starting with a marker such as `Sample 12:`. NeMo Curator filters work **per document**, so one big document would be kept or dropped as a whole. Splitting first lets every sample be judged on its own.

How it works:
1. `re.split(r"(?=Sample\s+\d+\s*:)", ...)` cuts the text *before* each `Sample N:` marker.
2. The marker is removed from the text and its number is stored as `sample_id`.
3. Each sample is written as one JSON line: `{"sample_id": ..., "text": ...}`.

> Note: this cell only reads the **first** record of the extracted file, so it expects a single PDF. If your PDFs use a different layout, adapt the regular expression.

In [47]:
import re
import json
from pathlib import Path

INPUT_FILE = EXTRACTED_JSONL_DIR / "pdf_extracted.jsonl"
OUTPUT_FILE = SPLIT_JSON_FILE

print(EXTRACTED_JSONL_DIR)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    record = json.loads(next(line for line in f if line.strip()))

full_text = record["text"]

# Find the position of the first actual sample marker to exclude the document header
first_sample_match = re.search(r"Sample\s+\d+\s*:", full_text)
if first_sample_match:
    # Slice full_text to keep only the text from the first sample marker onwards
    curated_text = full_text[first_sample_match.start():]
else:
    curated_text = full_text

# Split only the remaining sample text
samples = re.split(r"(?=Sample\s+\d+\s*:)", curated_text)
samples = [s.strip() for s in samples if s.strip()]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for i, sample in enumerate(samples, start=1):
        match = re.match(r"Sample\s+(\d+)\s*:\s*", sample)
        sample_id = int(match.group(1)) if match else i
        clean_text = re.sub(
            r"^Sample\s+\d+\s*:\s*", "", sample
        ).strip()

        f.write(json.dumps({
            "sample_id": sample_id,
            "text": clean_text
        }, ensure_ascii=False) + "\n")

        print(sample_id)
        print(clean_text)
        print(f"Sample {sample_id}: {len(clean_text.split())} words")
print("Samples created:", len(samples))
print("Output:", OUTPUT_FILE)

/content/drive/MyDrive/nemo_curator/data/extracted_jsonl
1
Information Retrieval
Information retrieval systems identify documents that are relevant to a user's
information need. Traditional approaches may rely on term frequencies and
inverted indexes, while neural approaches can represent queries and documents
using learned embeddings. Retrieval quality depends on indexing, query
formulation, document segmentation, and ranking. Precision at a chosen rank
measures how many retrieved items are relevant, whereas recall measures how
many relevant items were found. Hybrid systems combine lexical and semantic
signals to address different query types. Evaluation should reflect the
characteristics of the target collection. In practical applications, this issue should
be examined using documented procedures and representative examples.
Version 2 focuses on a different review pass: data collection, processing
consistency, evaluation, and limitations. Researchers should record assumptions,
compar

## Step 5 – Quality filtering and content cleaning

This is the core of the notebook. A NeMo Curator `Pipeline` is a chain of **stages**. Documents flow through the stages in order, and a document dropped by one filter never reaches the next one.

There are two kinds of stages:

- **Filters** (`ScoreFilter`) compute a score for each document and *remove* documents that fail a threshold.
- **Modifiers** (`Modify`) *rewrite* the text but never remove documents.

### Stages used (in execution order)

| # | Stage | Type | What it does | Setting used here |
|---|-------|------|--------------|-------------------|
| 1 | `JsonlReader` | Reader | Loads the split samples from JSONL | `SPLIT_JSON_FILE` |
| 2 | `WordCountFilter` | Filter | Removes documents that are too short or too long. Also saves the count in a new `word_count` field | `min_words=10`, `max_words=100000` |
| 3 | `UnicodeReformatter` | Modifier | Fixes broken or inconsistent Unicode (garbled characters, stray escape codes) using `ftfy` | defaults |
| 4 | `NewlineNormalizer` | Modifier | Replaces 3 or more consecutive newlines with exactly 2 | defaults |
| 5 | `NonAlphaNumericFilter` | Filter | Removes symbol-heavy text (e.g. `###@@@!!!`) | drop if more than 25% of characters are non-alphanumeric |
| 6 | `RepeatedLinesFilter` | Filter | Removes highly repetitive text. The score is *unique lines ÷ total lines* | keep only if score ≥ 0.7 |
| 7 | `PunctuationFilter` | Filter | Removes text without proper sentence endings | drop if more than 85% of sentences lack end punctuation |
| 8 | `BoilerPlateStringFilter` | Filter | Removes pages dominated by legal or cookie boilerplate (e.g. "privacy policy", "terms of use", "uses cookies") | drop if more than 40% of paragraphs match |
| 9 | `JsonlWriter` | Writer | Saves the surviving documents to `CURATED_JSONL_FILE` | – |

### Why this order?
- Cheap checks (word count) run first so later stages see fewer documents.
- Text is normalised (Unicode, newlines) *before* the character- and line-based filters, so they measure clean text rather than artefacts.

### Execution
`RayClient` starts a local Ray cluster, and `XennaExecutor` runs each stage in `batch` mode. The cluster is stopped at the end to free resources. The long log output is normal and shows each stage starting and finishing.

In [48]:
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.text.io.reader import JsonlReader
from nemo_curator.stages.text.io.writer import JsonlWriter
from nemo_curator.stages.text.filters import ScoreFilter


from nemo_curator.stages.text.filters.heuristic.repetition import RepeatedLinesFilter

from nemo_curator.stages.text.modifiers import Modify
from nemo_curator.stages.text.modifiers.unicode import UnicodeReformatter
from nemo_curator.stages.text.modifiers.string import NewlineNormalizer

from nemo_curator.stages.text.filters.heuristic import (
    WordCountFilter,
    NonAlphaNumericFilter,
    PunctuationFilter,
    BoilerPlateStringFilter
)

from nemo_curator.backends.xenna import XennaExecutor

# Start Ray client
ray_client = RayClient()
ray_client.start()

# Create processing pipeline
pipeline = Pipeline(name="quality_filtering")

# Load dataset - the starting point for all workflows
reader = JsonlReader(file_paths=str(SPLIT_JSON_FILE))
pipeline.add_stage(reader)

# Standard quality filtering pipeline (most common)
# Remove too short/long documents (essential)
# and save the word_count field
word_count_filter = ScoreFilter(
    filter_obj=WordCountFilter(min_words=10, max_words=100000),
    text_field="text",
    score_field="word_count"
)
pipeline.add_stage(word_count_filter)

# Normalize unicode characters (very common)
unicode_modifier = Modify(
    modifier_fn=UnicodeReformatter(),
    input_fields="text"
)
pipeline.add_stage(unicode_modifier)

#Standardize excessive line breaks
newline_modifier = Modify(
    modifier_fn=NewlineNormalizer(),
    input_fields="text"
)
pipeline.add_stage(newline_modifier)

# Remove symbol-heavy content
alpha_numeric_filter = ScoreFilter(
    filter_obj=NonAlphaNumericFilter(max_non_alpha_numeric_to_text_ratio=0.25),
    text_field="text"
)
pipeline.add_stage(alpha_numeric_filter)

# Remove repetitive content
repeated_lines_filter = ScoreFilter(
    filter_obj=RepeatedLinesFilter(max_repeated_line_fraction=0.7),
    text_field="text"
)
pipeline.add_stage(repeated_lines_filter)

# Ensure proper sentence structure
punctuation_filter = ScoreFilter(
    filter_obj=PunctuationFilter(max_num_sentences_without_endmark_ratio=0.85),
    text_field="text"
)
pipeline.add_stage(punctuation_filter)

#Remove template/boilerplate text
boilerplate_filter = ScoreFilter(
    filter_obj=BoilerPlateStringFilter(),
    text_field="text"
)
pipeline.add_stage(boilerplate_filter)

# Add writer stage
writer = JsonlWriter(str(CURATED_JSONL_FILE))
pipeline.add_stage(writer)

# Execute pipeline
executor = XennaExecutor(config={
    "execution_mode": "batch",
})
results = pipeline.run(executor)

# Cleanup Ray when done
ray_client.stop()

print("Task curator completed")


2026-10-19 05:51:59.662 | WARNING  | nemo_curator.core.client:start:121 - No monitoring services are running. Please run the `start_prometheus_grafana.py` script from nemo_curator/metrics folder to setup monitoring services separately.
2026-10-19 05:51:59.672 | INFO     | nemo_curator.core.utils:init_cluster:222 - Ray start command: ray start --head --node-ip-address 172.28.0.12 --port 6379 --metrics-export-port 8081 --dashboard-host 127.0.0.1 --dashboard-port 8265 --ray-client-server-port 10001 --temp-dir /tmp/ray --disable-usage-stats --block
2026-10-19 05:52:58.985 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'jsonl_reader' to pipeline 'quality_filtering'
2026-10-19 05:52:58.987 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'word_count' to pipeline 'quality_filtering'
2026-10-19 05:52:58.991 | INFO     | nemo_curator.pipeline.pipeline:add_stage:114 - Added stage 'UnicodeReformatter' to pipeline 'quality_filtering'
2026-10-19 05:52

Task curator completed


## Step 6 – Inspect the result

The writer may create either a single file or a folder of part files, so this cell handles both. It counts the records and prints the first three.

Each curated record contains:
- `sample_id` – the sample number from the original PDF
- `text` – the cleaned text
- `word_count` – added by `WordCountFilter`

### What to expect
The input had **77 samples** and **8 records** survived the pipeline, so the filters removed most of the intentionally noisy content. Compare a few kept and removed samples against the original PDF to check the thresholds suit your data.

**Tuning tips**
- Too many good documents removed → loosen the thresholds (lower `min_words`, raise `max_non_alpha_numeric_to_text_ratio`, etc.).
- Too much noise kept → tighten them, or add more filters from `nemo_curator.stages.text.filters`.
- Note that `BoilerPlateStringFilter` only recognises a fixed list of policy/cookie phrases. Spam such as "CLICK HERE TO SUBSCRIBE NOW" is **not** caught by it (the sample in record 3 above was kept), so you would need a custom filter for that kind of text.

In [49]:
import json
from pathlib import Path

# curated output path
output_path = Path(CURATED_JSONL_FILE)

# Find JSONL files, whether the writer produced a file or directory
if output_path.is_file():
    jsonl_files = [output_path]
elif output_path.is_dir():
    jsonl_files = sorted(output_path.rglob("*.jsonl"))
else:
    jsonl_files = []

if not jsonl_files:
    raise FileNotFoundError(
        f"No JSONL output found at: {output_path}\n"
        "Check the JsonlWriter output path and pipeline execution."
    )

print("Output path:", output_path)
print("JSONL files found:", len(jsonl_files))

# Count records and preview the first three
total_records = 0
preview_records = []

for file_path in jsonl_files:
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            total_records += 1

            if len(preview_records) < 50:
                preview_records.append(json.loads(line))

print("Total records:", total_records)

for i, record in enumerate(preview_records, start=1):
    print(f"\n{'=' * 70}")
    print(f"RECORD {i}")
    print("=" * 70)

    for key, value in record.items():
        if key == "text":
            print(f"\nText preview (first 2,000 characters):\n{value[:2000]}")
        else:
            print(f"{key}: {value}")

Output path: /content/drive/MyDrive/nemo_curator/data/curated/pdf_cleaned
JSONL files found: 1
Total records: 7

RECORD 1
sample_id: 6

Text preview (first 2,000 characters):
Data Quality Assessment
Data quality assessment examines whether a dataset is suitable for its intended
purpose. Dimensions can include completeness, consistency, validity,
uniqueness, timeliness, and representativeness. A missing field may be
acceptable in one application and unacceptable in another. Automated checks
can identify invalid values and structural anomalies, but domain experts may be
needed to assess semantic correctness. Quality reports should distinguish
observed defects from assumptions about what constitutes useful information.

Documenting these decisions makes the dataset easier to maintain and reuse. In
practical applications, this issue should be examined using documented
procedures and representative examples. Version 1 focuses on a different review
pass: data collection, processing consisten